Scenario : An inventory team wants to find products that have been out of stock for more than 30 days

Goal: Find products that are out of stock for more than 30 days.


In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, TimestampType
from datetime import datetime, timedelta

# Create the database/schema if it doesn't exist
spark.sql("CREATE DATABASE IF NOT EXISTS interview.25_sql")

# Drop the table if it already exists so the cell is re-runnable
spark.sql("DROP TABLE IF EXISTS interview.25_sql.inventory")

# Define the table schema
spark.sql("""
    CREATE TABLE interview.25_sql.inventory (
        product_id      INT,
        product_name    STRING,
        stock_quantity  INT,
        last_updated    TIMESTAMP
    )
""")

# Sample data: some products out of stock for more than 30 days, some recently restocked
base_time = datetime(2024, 1, 15, 10, 0, 0)

sample_data = [
    (1,  "Wireless Mouse",      0,  base_time - timedelta(days=45)),   # out of stock 45 days
    (2,  "Mechanical Keyboard", 0,  base_time - timedelta(days=60)),   # out of stock 60 days
    (3,  "USB-C Cable",         0,  base_time - timedelta(days=10)),   # out of stock 10 days
    (4,  "Monitor Stand",       12, base_time - timedelta(days=5)),    # in stock
    (5,  "Laptop Sleeve",        0,  base_time - timedelta(days=35)),   # out of stock 35 days
    (6,  "Webcam HD",            8,  base_time - timedelta(days=2)),    # in stock
    (7,  "Desk Lamp",            0,  base_time - timedelta(days=90)),   # out of stock 90 days
    (8,  "Headphones",           3,  base_time - timedelta(days=1)),    # in stock
    (9,  "Mousepad",             0,  base_time - timedelta(days=28)),   # out of stock 28 days
    (10, "Laptop Charger",       0,  base_time - timedelta(days=50)),   # out of stock 50 days
]

schema = StructType([
    StructField("product_id",     IntegerType(),  False),
    StructField("product_name",   StringType(),   False),
    StructField("stock_quantity", IntegerType(),  False),
    StructField("last_updated",   TimestampType(), False),
])

df = spark.createDataFrame(sample_data, schema)
df.write.mode("append").saveAsTable("interview.25_sql.inventory")

# Verify the data
display(spark.table("interview.25_sql.inventory"))

In [0]:
%sql
SELECT 
	product_id,product_name,last_updated,
	DATEDIFF(CURRENT_DATE,last_updated) AS days_diff
FROM interview.25_sql.inventory
WHERE stock_quantity =0
AND DATEDIFF(CURRENT_DATE,last_updated)>30;


Explanation:
We filter products with stock_qunatity=0 and use DATEDIFF() calculate the number of days since the last update. then we return products where the number of days is greater than 30